# 💰 Customer Acquisition Cost (CAC) & LTV Model
**Author:** Vincent Lepore | **Data Sources:** Google Ads API · Meta Ads API · HubSpot Deals · NetSuite ERP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/marketing-analytics-portfolio/blob/main/notebooks/03_cac_model.ipynb)

---

## Why CAC Matters
**Blended CAC = Total Spend / Total New Customers** — but this hides channel efficiency.

The real questions:
- Which channels acquire customers who **stay** (high LTV)?
- Are we paying $50 to acquire a $30 customer? (LTV:CAC < 1 = burning money)
- **Payback period**: how many months until a customer has paid back their acquisition cost?

### The LTV:CAC Benchmark
| Ratio | Signal | Action |
|-------|--------|--------|
| < 1x | Unprofitable | Pause channel immediately |
| 1–2x | At risk | Optimize or reduce spend |  
| 2–3x | Marginal | Monitor closely |
| 3x+ | Healthy | Scale with confidence |
| 5x+ | Outstanding | Maximize budget |

## 1. Setup & Data Loading

In [ ]:
!pip install numpy pandas matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

# Load data (run notebook 01 first)
df_gads = pd.read_csv("data/raw/google_ads_daily.csv")
df_meta = pd.read_csv("data/raw/meta_ads_daily.csv")
df_hs   = pd.read_csv("data/raw/hubspot_deals.csv")
df_ns   = pd.read_csv("data/raw/netsuite_transactions.csv")

print(f"Google Ads rows:  {len(df_gads):,}")
print(f"Meta Ads rows:    {len(df_meta):,}")
print(f"HubSpot deals:    {len(df_hs):,}")
print(f"NetSuite lines:   {len(df_ns):,}")

## 2. Monthly Spend Aggregation (Google Ads + Meta)

In [ ]:
# Google Ads: aggregate to monthly by channel group
df_gads['date']  = pd.to_datetime(df_gads['segments.date'])
df_gads['month'] = df_gads['date'].dt.to_period('M')
df_gads['channel_group'] = df_gads['campaign.name'].apply(
    lambda x: 'Brand Search'     if 'Brand' in str(x)
    else 'Non-Brand Search'  if any(w in str(x) for w in ['Generic','Competitor','Category'])
    else 'Remarketing'       if any(w in str(x) for w in ['Remarket','Retarget','RMK'])
    else 'Display / PMax'
)

gads_monthly = df_gads.groupby(['month','channel_group']).agg(
    spend       = ('metrics.cost', 'sum'),
    clicks      = ('metrics.clicks', 'sum'),
    conversions = ('metrics.conversions', 'sum'),
    conv_value  = ('metrics.conversions_value', 'sum'),
).reset_index()
gads_monthly['platform'] = 'Google Ads'

# Meta Ads: aggregate to monthly
df_meta['date']  = pd.to_datetime(df_meta['date_start'])
df_meta['month'] = df_meta['date'].dt.to_period('M')
df_meta['channel_group'] = df_meta['campaign_name'].apply(
    lambda x: 'Prospecting'  if any(w in str(x) for w in ['Prospect','Aware','PROS'])
    else 'Retargeting'    if any(w in str(x) for w in ['Retarget','RMK','LAL'])
    else 'Meta - Other'
)

meta_monthly = df_meta.groupby(['month','channel_group']).agg(
    spend       = ('spend', 'sum'),
    clicks      = ('clicks', 'sum'),
    conversions = ('actions.purchase', 'sum'),
    conv_value  = ('action_values.purchase', 'sum'),
).reset_index()
meta_monthly['platform'] = 'Meta Ads'

all_spend = pd.concat([
    gads_monthly.rename(columns={'channel_group':'channel'}),
    meta_monthly.rename(columns={'channel_group':'channel'}),
])
print(f"Monthly spend records: {len(all_spend):,}")
print(f"\nTotal spend by platform:")
print(all_spend.groupby('platform')['spend'].sum().apply(lambda x: f'${x:,.0f}'))

## 3. New Customers from HubSpot (Closed Won Deals)

In [ ]:
# Closed-won deals = new customers acquired
df_hs['closedate'] = pd.to_datetime(df_hs['closedate'], errors='coerce')
df_hs['month']     = df_hs['closedate'].dt.to_period('M')

SOURCE_MAP = {
    'ORGANIC_SEARCH': 'Non-Brand Search', 'PAID_SEARCH': 'Non-Brand Search',
    'SOCIAL_MEDIA': 'Prospecting',        'PAID_SOCIAL': 'Prospecting',
    'EMAIL_MARKETING': 'Email',           'REFERRAL': 'Referral',
    'DIRECT_TRAFFIC': 'Direct',           'OTHER_CAMPAIGNS': 'Other',
}

won_deals = df_hs[df_hs['hs_deal_is_closed_won'] == True].copy()
won_deals['channel'] = won_deals['hs_analytics_source'].map(SOURCE_MAP).fillna('Other')
won_deals['hs_closed_amount'] = pd.to_numeric(won_deals['hs_closed_amount'], errors='coerce').fillna(0)

new_cust_monthly = won_deals.groupby('month').agg(
    new_customers = ('hs_deal_id', 'count'),
    new_arr       = ('hs_closed_amount', 'sum'),
    avg_deal_size = ('hs_closed_amount', 'mean'),
    avg_days_close= ('days_to_close', 'mean'),
).reset_index()

print(f"Months of data: {len(new_cust_monthly)}")
print(f"Total new customers: {new_cust_monthly['new_customers'].sum():,}")
print(f"Total ARR: ${new_cust_monthly['new_arr'].sum():,.0f}")
print(f"Avg deal size: ${new_cust_monthly['avg_deal_size'].mean():,.0f}")
print(f"Avg days to close: {new_cust_monthly['avg_days_close'].mean():.0f}")

## 4. LTV Calculation from NetSuite

In [ ]:
df_ns['TranDate'] = pd.to_datetime(df_ns['TranDate'], format='%m/%d/%Y', errors='coerce')
df_ns['month']    = df_ns['TranDate'].dt.to_period('M')
df_ns['Amount']   = pd.to_numeric(df_ns['Amount'], errors='coerce').fillna(0)

# Revenue per customer per month
cust_rev = df_ns[df_ns['Amount'] > 0].groupby(['Entity','month'])['Amount'].sum().reset_index()

# First month for each customer (acquisition cohort)
first_month = cust_rev.groupby('Entity')['month'].min().reset_index()
first_month.columns = ['Entity','cohort_month']
cust_rev = cust_rev.merge(first_month, on='Entity')
cust_rev['months_since_acq'] = (
    cust_rev['month'].apply(lambda x: x.ordinal) - 
    cust_rev['cohort_month'].apply(lambda x: x.ordinal)
)

# LTV at 12 and 24 months
ltv_12 = cust_rev[cust_rev['months_since_acq'] < 12].groupby('Entity')['Amount'].sum()
ltv_24 = cust_rev[cust_rev['months_since_acq'] < 24].groupby('Entity')['Amount'].sum()

avg_ltv_12 = ltv_12.mean()
avg_ltv_24 = ltv_24.mean()
print(f"Average LTV 12-month: ${avg_ltv_12:,.2f}")
print(f"Average LTV 24-month: ${avg_ltv_24:,.2f}")
print(f"LTV expansion (12→24m): {avg_ltv_24/avg_ltv_12:.2f}x")

## 5. CAC by Channel + Health Scoring

In [ ]:
# Channel-level CAC from ad platforms
channel_cac = all_spend.groupby(['platform','channel']).agg(
    spend       = ('spend', 'sum'),
    conversions = ('conversions', 'sum'),
    conv_value  = ('conv_value', 'sum'),
    clicks      = ('clicks', 'sum'),
).reset_index()

channel_cac['cac']  = (channel_cac['spend'] / channel_cac['conversions'].replace(0, np.nan)).round(2)
channel_cac['roas'] = (channel_cac['conv_value'] / channel_cac['spend']).round(3)
channel_cac['cvr']  = (channel_cac['conversions'] / channel_cac['clicks'].replace(0, np.nan)).round(4)

# LTV multipliers by channel (brand search acquires higher-quality customers)
LTV_MULT = {
    'Brand Search':2.8, 'Non-Brand Search':2.5, 'Remarketing':2.9,
    'Display / PMax':1.7, 'Prospecting':1.8, 'Retargeting':2.5, 'Meta - Other':1.5,
}
channel_cac['ltv_24m']      = channel_cac.apply(lambda r: r['cac'] * LTV_MULT.get(r['channel'], 2.0), axis=1)
channel_cac['ltv_cac_ratio']= (channel_cac['ltv_24m'] / channel_cac['cac'].replace(0, np.nan)).round(2)
channel_cac['payback_months']= (channel_cac['cac'] / (channel_cac['ltv_24m'] / 24)).round(1)

def health(ratio):
    if ratio >= 3: return '✅ Healthy'
    elif ratio >= 2: return '⚠️ Marginal'
    elif ratio >= 1: return '🔶 At Risk'
    else: return '🔴 Unprofitable'

channel_cac['health'] = channel_cac['ltv_cac_ratio'].apply(health)

print("\nChannel CAC Health Matrix:")
print(f"{'Channel':<22} {'Platform':<12} {'CAC':>8} {'LTV 24m':>10} {'LTV:CAC':>8} {'Payback':>9} {'Status'}")
print("-"*80)
for _, r in channel_cac.sort_values('ltv_cac_ratio', ascending=False).iterrows():
    print(f"  {r['channel']:<20} {r['platform']:<12} ${r['cac']:>6,.0f} ${r['ltv_24m']:>8,.0f} {r['ltv_cac_ratio']:>7.1f}x {r['payback_months']:>7.1f}mo  {r['health']}")

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Customer Acquisition Cost Analysis', fontsize=16, fontweight='bold', color='white', y=0.98)

HEALTH_COLORS = {'✅ Healthy':'#00f5d4','⚠️ Marginal':'#fee440','🔶 At Risk':'#f4a261','🔴 Unprofitable':'#e63946'}
PLATFORM_COLORS = {'Google Ads':'#00b4d8','Meta Ads':'#9b5de5'}

# 1. CAC vs LTV bar chart
ax = axes[0,0]; ax.set_facecolor('#161b22')
sorted_cac = channel_cac.sort_values('ltv_24m', ascending=True)
y = np.arange(len(sorted_cac))
ax.barh(y - 0.2, sorted_cac['cac'],    0.35, label='CAC',    color='#f4a261', alpha=0.9)
ax.barh(y + 0.2, sorted_cac['ltv_24m'],0.35, label='LTV 24m',color='#00b4d8', alpha=0.9)
ax.set_yticks(y); ax.set_yticklabels([f"{r['channel'][:18]}
({r['platform'][:6]})" for _,r in sorted_cac.iterrows()], fontsize=8)
ax.set_xlabel('Amount ($)'); ax.legend(fontsize=9)
ax.set_title('CAC vs 24-Month LTV by Channel', fontweight='bold')
ax.grid(axis='x', alpha=0.3, color='white')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# 2. LTV:CAC ratio with 3x benchmark
ax = axes[0,1]; ax.set_facecolor('#161b22')
ratios = channel_cac.sort_values('ltv_cac_ratio', ascending=True)
colors = [HEALTH_COLORS[h] for h in ratios['health']]
ax.barh(range(len(ratios)), ratios['ltv_cac_ratio'], color=colors, alpha=0.85)
ax.axvline(3, color='white', linestyle='--', linewidth=1.2, label='3x Target')
ax.set_yticks(range(len(ratios)))
ax.set_yticklabels([r['channel'] for _,r in ratios.iterrows()], fontsize=9)
ax.set_xlabel('LTV:CAC Ratio (x)'); ax.legend(fontsize=9)
ax.set_title('LTV:CAC Ratio — Channel Health', fontweight='bold')
ax.grid(axis='x', alpha=0.3, color='white')

# 3. Payback period
ax = axes[1,0]; ax.set_facecolor('#161b22')
pb = channel_cac.sort_values('payback_months', ascending=True)
ax.barh(range(len(pb)), pb['payback_months'],
        color=[PLATFORM_COLORS.get(p,'#adb5bd') for p in pb['platform']], alpha=0.85)
ax.axvline(12, color='#fee440', linestyle='--', linewidth=1.2, label='12-month target')
ax.set_yticks(range(len(pb)))
ax.set_yticklabels([r['channel'] for _,r in pb.iterrows()], fontsize=9)
ax.set_xlabel('Payback Period (Months)'); ax.legend(fontsize=9)
ax.set_title('CAC Payback Period by Channel', fontweight='bold')
ax.grid(axis='x', alpha=0.3, color='white')

# 4. CAC trend
ax = axes[1,1]; ax.set_facecolor('#161b22')
spend_monthly = all_spend.groupby('month')['spend'].sum().reset_index()
cust_monthly  = new_cust_monthly.copy()
spend_monthly['month_str'] = spend_monthly['month'].astype(str)
cust_monthly['month_str']  = cust_monthly['month'].astype(str)
merged = spend_monthly.merge(cust_monthly[['month_str','new_customers']], on='month_str', how='inner')
merged['cac'] = merged['spend'] / merged['new_customers']
ax.plot(range(len(merged)), merged['cac'], color='#f4a261', linewidth=2, marker='o', markersize=4)
ax.fill_between(range(len(merged)), merged['cac'], alpha=0.2, color='#f4a261')
ax.set_xlabel('Month Index'); ax.set_ylabel('Blended CAC ($)')
ax.set_title('Blended CAC Trend (All Channels)', fontweight='bold')
ax.grid(alpha=0.3, color='white')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.tight_layout(pad=3)
plt.savefig('data/processed/cac_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()